# Benchmark 2 mock test: five paper phases + one unclassified

This notebook stress-tests the current `form_factor.py` and `kagome_order_diagnosis.py`
using synthetic kernels.

Targets:
- FM
- PI
- cBO (one-`Q` component)
- sBO (one-`Q` component)
- f-SC
- one deliberately new / unmatched mode, which should come back as `unclassified`

These are **mock recognition tests**, not full FRG-flow physics tests.


In [2]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path('/mnt/data') if Path('/mnt/data/form_factor.py').exists() else Path('.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from form_factor import OrderRecognizer
from kagome_order_diagnosis import KagomeOrderDiagnoser

print('Using files from:', ROOT)


Using files from: .


In [3]:
class DummyPatch:
    def __init__(self, k_cart, eigvec):
        self.k_cart = np.asarray(k_cart, dtype=float)
        self.eigvec = np.asarray(eigvec, dtype=complex)


class DummyPatchSet:
    def __init__(self, patches):
        self.patches = list(patches)


class DummyKernel:
    def __init__(self, name, Q, matrix, partner, col_spins=('up', 'up')):
        self.name = name
        self.Q = np.asarray(Q, dtype=float)
        self.matrix = np.asarray(matrix, dtype=complex)
        self.col_partner_patches = np.asarray(partner, dtype=int)
        self.col_spins = tuple(col_spins)

    def eig(self, sort_by='abs'):
        vals, vecs = np.linalg.eigh(self.matrix)
        if sort_by == 'abs':
            idx = np.argsort(np.abs(vals))[::-1]
        else:
            idx = np.argsort(vals)[::-1]
        return vals[idx], vecs[:, idx]


def onehot(i, n=3):
    v = np.zeros(n, dtype=float)
    v[i] = 1.0
    return v


def ring_ks(n_patch=12, radius=1.0):
    ang = np.linspace(0.0, 2.0 * np.pi, n_patch, endpoint=False)
    return np.column_stack([radius * np.cos(ang), radius * np.sin(ang)])


def kernel_from_modes(name, Q, modes, partner, col_spins=('up', 'up'), weights=None):
    modes = [np.asarray(v, dtype=complex) for v in modes]
    if weights is None:
        weights = [1.0] * len(modes)
    n = len(modes[0])
    mat = np.zeros((n, n), dtype=complex)
    for w, v in zip(weights, modes):
        nrm = np.linalg.norm(v)
        if nrm < 1e-14:
            continue
        v = v / nrm
        mat += w * np.outer(v, v.conjugate())
    return DummyKernel(name=name, Q=Q, matrix=mat, partner=partner, col_spins=col_spins)


def build_patchsets_patterned(n_patch=12):
    ks = ring_ks(n_patch)
    labels = np.array([i % 3 for i in range(n_patch)], dtype=int)  # A,B,C cycling
    patches_up = [DummyPatch(k, onehot(labels[i])) for i, k in enumerate(ks)]
    # keep dn different just to exercise spin-aware machinery, though most ph tests use uu
    patches_dn = [DummyPatch(k, onehot((labels[i] + 1) % 3)) for i, k in enumerate(ks)]
    return {'up': DummyPatchSet(patches_up), 'dn': DummyPatchSet(patches_dn)}, ks, labels


def build_patchsets_fsc(n_patch=12):
    ks = ring_ks(n_patch)
    # up leg lives on A, dn leg lives on C -> strong AC triplet template overlap
    patches_up = [DummyPatch(k, onehot(0)) for k in ks]
    patches_dn = [DummyPatch(k, onehot(2)) for k in ks]
    return {'up': DummyPatchSet(patches_up), 'dn': DummyPatchSet(patches_dn)}, ks


Q1 = np.array([-0.5, -np.sqrt(3.0) / 2.0])
Q2 = np.array([1.0, 0.0])
Q3 = np.array([-0.5,  np.sqrt(3.0) / 2.0])

PAIR_LABELS = {0: 'A', 1: 'B', 2: 'C'}


## Build a patterned patch basis for FM / PI / cBO / sBO / novel

For finite-`Q` bond-order mocks, we will use a partner map that turns the leading reconstructed
pair into `A-C`, which matches the paper-template choice associated with `Q2` in the current
diagnoser implementation.


In [5]:
patchsets_ph, ks, labels = build_patchsets_patterned(n_patch=12)
diag_ph = KagomeOrderDiagnoser(patchsets_by_spin=patchsets_ph)

N = len(ks)
identity = np.arange(N)
A_idx = np.where(labels == 0)[0]
B_idx = np.where(labels == 1)[0]
C_idx = np.where(labels == 2)[0]

partner_AC = np.array(identity)
for j, p in enumerate(A_idx):
    partner_AC[p] = C_idx[j % len(C_idx)]
for p in np.where(labels != 0)[0]:
    partner_AC[p] = C_idx[0]

kx = ks[:, 0]
ky = ks[:, 1]
theta = np.arctan2(ky, kx)

f_dx2y2 = np.cos(2 * kx) - np.cos(kx) * np.cos(np.sqrt(3.0) * ky)
f_dxy = np.sqrt(3.0) * np.sin(kx) * np.sin(np.sqrt(3.0) * ky)

# one-Q bond-order mock: only support A-sector patches so that partner map reconstructs A-C bonds
g_bo = np.zeros(N, dtype=float)
g_bo[A_idx] = np.sin(ks[A_idx] @ Q2)

# deliberately unmatched p-wave-like particle-hole mode
g_novel = np.cos(theta)

kernels_ph = {
    'FM': kernel_from_modes('ph_spin_uu', np.zeros(2), [np.ones(N)], identity),
    'PI': kernel_from_modes('ph_charge_uu', np.zeros(2), [f_dx2y2, f_dxy], identity, weights=[1.0, 1.0]),
    'cBO': kernel_from_modes('ph_charge_uu', Q2, [g_bo], partner_AC),
    'sBO': kernel_from_modes('ph_spin_uu',   Q2, [g_bo], partner_AC),
    'Novel': kernel_from_modes('ph_charge_uu', np.zeros(2), [g_novel], identity),
}

results_ph = {name: diag_ph.diagnose_kernel(kernel) for name, kernel in kernels_ph.items()}


## Build a dedicated spin-aware patch basis for f-SC

Here the mock is designed so that the reconstructed triplet pairing lives strongly in the `A-C`
sublattice pair, which the current paper-template layer recognizes as one of the kagome f-wave
components.


In [7]:
patchsets_sc, ks_sc = build_patchsets_fsc(n_patch=12)
diag_sc = KagomeOrderDiagnoser(patchsets_by_spin=patchsets_sc)

kx_sc = ks_sc[:, 0]
ky_sc = ks_sc[:, 1]
f_ac = np.sin(np.sqrt(3.0) * ky_sc)

kernel_fsc = kernel_from_modes(
    'pp_triplet_sz0',
    np.zeros(2),
    [f_ac],
    np.arange(len(ks_sc)),
    col_spins=('up', 'dn'),
)
result_fsc = diag_sc.diagnose_kernel(kernel_fsc)


In [8]:
rows = []
for name in ['FM', 'PI', 'cBO', 'sBO', 'Novel']:
    r = results_ph[name]
    rows.append({
        'mock_case': name,
        'expected': 'unclassified' if name == 'Novel' else name,
        'paper_label': r.paper_label,
        'coarse_label': r.coarse_label,
        'top_template': r.top_template_name,
        'top_template_score': round(r.top_template_score, 6),
        'dominant_pair': None if r.internal_mode.dominant_pair is None else f"{PAIR_LABELS[r.internal_mode.dominant_pair[0]]}-{PAIR_LABELS[r.internal_mode.dominant_pair[1]]}",
        'same_weight': round(r.internal_mode.same_sublattice_weight, 6),
        'inter_weight': round(r.internal_mode.inter_sublattice_weight, 6),
    })

rows.append({
    'mock_case': 'fSC',
    'expected': 'f-SC',
    'paper_label': result_fsc.paper_label,
    'coarse_label': result_fsc.coarse_label,
    'top_template': result_fsc.top_template_name,
    'top_template_score': round(result_fsc.top_template_score, 6),
    'dominant_pair': None if result_fsc.internal_mode.dominant_pair is None else f"{PAIR_LABELS[result_fsc.internal_mode.dominant_pair[0]]}-{PAIR_LABELS[result_fsc.internal_mode.dominant_pair[1]]}",
    'same_weight': round(result_fsc.internal_mode.same_sublattice_weight, 6),
    'inter_weight': round(result_fsc.internal_mode.inter_sublattice_weight, 6),
})

df = pd.DataFrame(rows)
df


,mock_case,expected,paper_label,coarse_label,top_template,top_template_score,dominant_pair,same_weight,inter_weight
0,FM,FM,FM,s,FM_constant,0.333333,C-C,1.0,0.0
1,PI,PI,PI,paper_d_Q0,PI_dx2y2,0.333333,C-C,1.0,0.0
2,cBO,cBO,cBO,p,BO_finiteQ,0.152056,A-C,0.0,1.0
3,sBO,sBO,sBO,p,BO_finiteQ,0.152056,A-C,0.0,1.0
4,Novel,unclassified,unclassified,p,PI_dx2y2,0.000000,A-A,1.0,0.0
5,fSC,f-SC,f-SC,pp_odd,fSC_AC,1.000000,A-C,0.0,1.0


In [9]:
expected = {
    'FM': 'FM',
    'PI': 'PI',
    'cBO': 'cBO',
    'sBO': 'sBO',
    'fSC': 'f-SC',
    'Novel': 'unclassified',
}

observed = {name: results_ph[name].paper_label for name in ['FM', 'PI', 'cBO', 'sBO', 'Novel']}
observed['fSC'] = result_fsc.paper_label

for key, want in expected.items():
    got = observed[key]
    print(f'{key:>6s}: expected={want:<12s} observed={got}')
    assert got == want, f'{key}: expected {want}, got {got}'

print('\nAll mock Benchmark2 phase-recognition checks passed.')


    FM: expected=FM           observed=FM
    PI: expected=PI           observed=PI
   cBO: expected=cBO          observed=cBO
   sBO: expected=sBO          observed=sBO
   fSC: expected=f-SC         observed=f-SC
 Novel: expected=unclassified observed=unclassified

All mock Benchmark2 phase-recognition checks passed.


## Inspect a few detailed summaries

These summaries are useful for checking *why* each mock was classified the way it was.


In [11]:
for name in ['FM', 'PI', 'cBO', 'sBO', 'Novel']:
    res = results_ph[name]
    print('=' * 90)
    print(name)
    print(res.summary_dict())

print('=' * 90)
print('fSC')
print(result_fsc.summary_dict())


FM
{'kernel': 'ph_spin_uu', 'Q': [0.0, 0.0], 'coarse_label': 's', 'coarse_score': 1.0, 'paper_label': 'FM', 'paper_score': 0.3333333333333333, 'recognition_status': 'recognized', 'top_template_name': 'FM_constant', 'top_template_score': 0.3333333333333333, 'spin_sector': 'spin/triplet', 'channel_sector': 'ph', 'leg_spin_structure': ('up', 'up'), 'degeneracy': 1, 'internal_mode': {'channel_sector': 'ph', 'spin_sector': 'spin/triplet', 'leg_spin_structure': ('up', 'up'), 'tensor_basis_shape': (12, 3, 3, 1), 'dominant_pair': (2, 2), 'same_sublattice_weight': 1.0, 'inter_sublattice_weight': 0.0, 'pair_weights': [[0.3333333333333333, 0.0, 0.0], [0.0, 0.3333333333333333, 0.0], [0.0, 0.0, 0.33333333333333337]]}, 'template_scores': [{'name': 'FM_constant', 'score': 0.3333333333333333}, {'name': 'PI_dx2y2', 'score': 8.137054015036071e-33}, {'name': 'PI_dxy', 'score': 8.925310006919372e-34}, {'name': 'fSC_AB', 'score': 0.0}, {'name': 'fSC_BC', 'score': 0.0}, {'name': 'fSC_AC', 'score': 0.0}], 'n